## Feature Engineering & Split Data
---
Input  : `data_cleaned_binary.csv`  
Output : `data_engineered_binary.csv`, `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`

### 2.1 Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
print('✅ Library berhasil diimport')

### 2.2 Load Data Cleaned (Biner)

In [ ]:
df = pd.read_csv(r'E:\Python - Project\phone-addiction-detection\outputs\Pre-process-out\data_cleaned_binary.csv')
print(f'Shape data: {df.shape}')
print(f'Distribusi target Addicted:\n{df["Addicted"].value_counts()}')
df.head(5)

### 2.3 Analisis Korelasi Pearson (terhadap label biner)

In [ ]:
# Korelasi semua fitur numerik terhadap target biner
num_cols = ['App Usage Time (min/day)', 'Screen On Time (hours/day)',
            'Battery Drain (mAh/day)', 'Number of Apps Installed',
            'Data Usage (MB/day)', 'Age', 'Gender', 'Operating System', 'Addicted']

corr_matrix = df[num_cols].corr(method='pearson')

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={'size': 10})
plt.title('Matriks Korelasi Pearson – Fitur vs Target Biner (Addicted)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('02_korelasi_pearson_biner.png', dpi=150, bbox_inches='tight')
plt.show()

# Tampilkan korelasi terhadap target
corr_target = corr_matrix['Addicted'].drop('Addicted').sort_values(ascending=False)
print('=== Korelasi Fitur terhadap Label Biner (Addicted) ===')
for col, val in corr_target.items():
    bar = '█' * int(abs(val) * 30)
    print(f'  {col:<40}: {val:+.4f}  {bar}')

### 2.4 Feature Importance – Mutual Information

In [ ]:
X_temp = df.drop(columns=['Addicted'])
y_temp = df['Addicted']

mi_scores = mutual_info_classif(X_temp, y_temp, random_state=42)
mi_df = pd.DataFrame({'Feature': X_temp.columns, 'MI Score': mi_scores})\
          .sort_values('MI Score', ascending=False).reset_index(drop=True)

plt.figure(figsize=(12, 6))
colors_mi = ['#e74c3c' if s > 0.1 else '#3498db' for s in mi_df['MI Score']]
plt.barh(mi_df['Feature'], mi_df['MI Score'], color=colors_mi, edgecolor='black')
plt.xlabel('Mutual Information Score')
plt.title('Mutual Information – Feature Importance terhadap Label Biner', fontweight='bold')
plt.gca().invert_yaxis()
plt.axvline(0.1, color='orange', linestyle='--', label='Threshold MI = 0.1')
plt.legend()
plt.tight_layout()
plt.savefig('02_mutual_info_biner.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== Mutual Information Score ===')
print(mi_df.to_string(index=False))

### 2.5 Feature Selection

> Fitur yang dipilih berdasarkan gabungan korelasi Pearson (|r| > 0.05) dan  
> Mutual Information (MI > 0.01). Fitur dummy Device Model diabaikan karena MI sangat rendah.

In [ ]:
# Fitur inti dengan korelasi & MI tinggi terhadap target
selected_features = [
    'App Usage Time (min/day)',
    'Screen On Time (hours/day)',
    'Battery Drain (mAh/day)',
    'Number of Apps Installed',
    'Data Usage (MB/day)',
    'Age',
    'Gender',
    'Operating System'
]

# Verifikasi semua fitur tersedia
selected_features = [f for f in selected_features if f in df.columns]

X = df[selected_features].copy()
y = df['Addicted'].copy()

print(f'Fitur yang dipilih ({len(selected_features)}):')
for f in selected_features:
    print(f'  ✅ {f}')
print(f'\nShape X: {X.shape}')
print(f'Distribusi y:\n{y.value_counts()}')

### 2.6 Feature Engineering – Tambah Fitur Interaksi

> Dua fitur baru ditambahkan berdasarkan domain knowledge:  
> - **Usage_Intensity** = App Usage Time × Battery Drain (proxy intensitas keseluruhan)  
> - **Screen_Data_Ratio** = Screen On Time / Data Usage (efisiensi pemakaian data per jam layar)

In [ ]:
df_eng = df[selected_features + ['Addicted']].copy()

# Fitur interaksi 1: Intensitas penggunaan
df_eng['Usage_Intensity'] = (
    df_eng['App Usage Time (min/day)'] * df_eng['Battery Drain (mAh/day)']
) / 1e5  # Normalisasi skala

# Fitur interaksi 2: Rasio layar terhadap data
df_eng['Screen_Data_Ratio'] = (
    df_eng['Screen On Time (hours/day)'] /
    (df_eng['Data Usage (MB/day)'] + 1)  # +1 menghindari divisi nol
)

# Update selected features
selected_features_eng = selected_features + ['Usage_Intensity', 'Screen_Data_Ratio']

print(f'Fitur setelah Feature Engineering ({len(selected_features_eng)}):')
for f in selected_features_eng:
    print(f'  ✅ {f}')

# Statistik fitur baru
print('\n=== Statistik Fitur Baru ===')
print(df_eng[['Usage_Intensity', 'Screen_Data_Ratio']].describe().round(4))

In [ ]:
# Visualisasi fitur baru per kelas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_bin = ['#2ecc71', '#e74c3c']

for i, col in enumerate(['Usage_Intensity', 'Screen_Data_Ratio']):
    for label, color, name in zip([0,1], colors_bin, ['Tidak Kecanduan','Kecanduan']):
        axes[i].hist(df_eng[df_eng['Addicted']==label][col],
                     bins=30, alpha=0.6, color=color, label=name, edgecolor='white')
    axes[i].set_title(f'Distribusi: {col}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frekuensi')
    axes[i].legend()

plt.suptitle('Fitur Interaksi Baru – Distribusi per Label Biner', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('02_fitur_interaksi.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.7 Normalisasi MinMax Scaler

In [ ]:
X = df_eng[selected_features_eng].copy()
y = df_eng['Addicted'].copy()

scaler  = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print('✅ Normalisasi MinMax selesai')
print('Statistik setelah normalisasi (semua nilai dalam [0, 1]):')
X_scaled.describe().round(3)

### 2.8 Split Data Train & Test (80:20, Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print('=== Pembagian Data (80:20 Stratified) ===')
print(f'Total data  : {len(X_scaled)}')
print(f'Data train  : {len(X_train)} ({len(X_train)/len(X_scaled)*100:.0f}%)')
print(f'Data test   : {len(X_test)}  ({len(X_test)/len(X_scaled)*100:.0f}%)')

print(f'\nDistribusi Train:')
print(f'  0 (Tidak Kecanduan): {(y_train==0).sum()} ({(y_train==0).sum()/len(y_train)*100:.1f}%)')
print(f'  1 (Kecanduan)      : {(y_train==1).sum()} ({(y_train==1).sum()/len(y_train)*100:.1f}%)')
print(f'\nDistribusi Test:')
print(f'  0 (Tidak Kecanduan): {(y_test==0).sum()} ({(y_test==0).sum()/len(y_test)*100:.1f}%)')
print(f'  1 (Kecanduan)      : {(y_test==1).sum()} ({(y_test==1).sum()/len(y_test)*100:.1f}%)')

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors_bin = ['#2ecc71', '#e74c3c']

for ax, y_data, title in zip(axes, [y_train, y_test], ['Train Set (80%)', 'Test Set (20%)']):
    counts = y_data.value_counts().sort_index()
    ax.bar(['0 – Tidak Kecanduan', '1 – Kecanduan'], counts.values,
           color=colors_bin, edgecolor='black', width=0.4)
    ax.set_title(f'Distribusi Kelas – {title}', fontweight='bold')
    ax.set_ylabel('Jumlah')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 3, str(v), ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('02_distribusi_split_biner.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualisasi split disimpan.')

### 2.9 Simpan Semua Output

In [ ]:
df_eng.to_csv('data_engineered_binary.csv', index=False)
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv',   index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv',   index=False)

joblib.dump(scaler,              'minmax_scaler.pkl')
joblib.dump(selected_features_eng, 'selected_features.pkl')

print('✅ File berhasil disimpan:')
print('   - data_engineered_binary.csv')
print('   - X_train.csv | X_test.csv')
print('   - y_train.csv | y_test.csv')
print('   - minmax_scaler.pkl')
print('   - selected_features.pkl')
print(f'\nTotal fitur akhir : {len(selected_features_eng)}')
print(f'Total sampel      : {len(df_eng)}')